In [ ]:
from pathlib import Path
import os

# Keep relative input and output paths anchored at the project root.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "experiments" else Path.cwd()
if not (PROJECT_ROOT / "utils.py").is_file():
    raise RuntimeError(f"Open this notebook from the project root or experiments/: {Path.cwd()}")
os.chdir(PROJECT_ROOT)

!pip install -r requirements.txt


In [ ]:
from segment_anything import sam_model_registry, SamPredictor
from utils import download_SAM_weights
import torch
import cv2
import numpy as np

Generate a mask via SAM for the boundary box and image

In [ ]:
# load SAM weights
SAM_weights = download_SAM_weights()
model_type = "vit_h"
device = "cuda" if torch.cuda.is_available() else "cpu"

# load SAM to RAM
sam = sam_model_registry[model_type](checkpoint=SAM_weights)
sam.to(device=device)
predictor = SamPredictor(sam)

# setup the current image
image_path = "test_images/test_image_1.jpg"
image = cv2.imread(image_path)
image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

input_image = predictor.transform.apply_image(image)
input_image_torch = torch.as_tensor(input_image, device=device, dtype=torch.float32)
input_image_torch = input_image_torch.permute(2, 0, 1).contiguous()[None, :, :, :]
predictor.set_torch_image(input_image_torch, image.shape[:2])

# define the points for the bounding box
x1, y1 = 100, 250  # left upper
x2, y2 = 600, 1000  # down right
x1, y1 = 100, 650  # left upper
x2, y2 = 240, 1000  # down right

# form the box
input_box = np.array([x1, y1, x2, y2])

# start segmentation for given image on the bounding box
masks, scores, logits = predictor.predict(
    point_coords=None,
    point_labels=None,
    box=input_box,
    multimask_output=False  # False выдает одну самую точную маску для бокса
)

# masks — numpy array of booleans (1, H, W)
mask_tensor = torch.tensor(masks[0])
print(f"Размер маски: {mask_tensor.shape}, количество пикселей объекта: {mask_tensor.sum().item()}")

Wrap segmentation in separate function

In [ ]:
def segment_image(image_path: str, x1, y1, x2, y2):
    SAM_weights = download_SAM_weights()
    model_type = "vit_h"
    device = "cuda" if torch.cuda.is_available() else "cpu"

    # load SAM to RAM
    sam = sam_model_registry[model_type](checkpoint=SAM_weights)
    sam.to(device=device)
    predictor = SamPredictor(sam)

    # setup the current image
    image = cv2.imread(image_path)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    input_image = predictor.transform.apply_image(image)
    input_image_torch = torch.as_tensor(input_image, device=device, dtype=torch.float32)
    input_image_torch = input_image_torch.permute(2, 0, 1).contiguous()[None, :, :, :]
    predictor.set_torch_image(input_image_torch, image.shape[:2])

    # form the box
    input_box = np.array([x1, y1, x2, y2])

    # start segmentation for given image on the bounding box
    masks, scores, logits = predictor.predict(
        point_coords=None,
        point_labels=None,
        box=input_box,
        multimask_output=False # output one mask instead of many masks
    )

    # masks — numpy array of booleans (1, H, W)
    return masks[0]


In [ ]:
from utils import segment_image

Visualisation test

In [ ]:
import matplotlib.pyplot as plt

# mask = masks[0]
mask = segment_image("test_images/test_image_1.jpg", 100, 700, 500, 1000)

plt.figure(figsize=(10, 10))
plt.imshow(image)  # original image

# apply the mask
overlay = np.zeros((*mask.shape, 4), dtype=np.float32)
overlay[mask] = [1.0, 0.0, 0.0, 0.5]  # Color of the mask
plt.imshow(overlay)

# Show bounding box
plt.gca().add_patch(plt.Rectangle((x1, y1), x2 - x1, y2 - y1,
                                  edgecolor='yellow', facecolor='none', linewidth=2))

plt.title("Output of SAM including bounding box")
plt.axis('off')
plt.show()